In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)


def generate_transaction():
    tx_id = f"TX{random.randint(1000, 9999)}"
    user_id = f"u{random.randint(1, 20):02d}"
    amount = round(random.uniform(5.0, 5000.0), 2)
    store = random.choice(["Warszawa", "Kraków", "Gdańsk", "Wrocław"])
    category = random.choice(["elektronika", "odzież", "żywność", "książki"])
    timestamp = datetime.now().isoformat()

    return {
        "tx_id": tx_id,
        "user_id": user_id,
        "amount": amount,
        "store": store,
        "category": category,
        "timestamp": timestamp
    }


while True:
    transakcja = generate_transaction()
    producer.send('transactions', value=transakcja)
    print(transakcja)
    time.sleep(1)

Writing producer.py


In [2]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    dane = message.value
    
    if dane['amount'] > 3000:
        print(f"ALERT: {dane['tx_id']} | {dane['amount']} PLN | {dane['store']} | {dane['category']}")

Writing consumer_filter.py


In [3]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='grupa_oceniajaca_ryzyko',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    dane = message.value 
    kwota = dane['amount']
    
    if kwota > 3000:
        dane['risk_level'] = "HIGH"
    elif kwota > 1000:
        dane['risk_level'] = "MEDIUM"
    else:
        dane['risk_level'] = "LOW"
        
    print(dane)

Writing consumer_enrich.py


In [4]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0


for message in consumer:
    dane = message.value
    sklep = dane['store']
    kwota = dane['amount']
    

    store_counts[sklep] += 1
     
    total_amount[sklep] = total_amount.get(sklep, 0) + kwota
    
    msg_count += 1
    
    if msg_count % 10 == 0:
        print(f"\n--- PODSUMOWANIE (po {msg_count} transakcjach) ---")
        print("Sklep | Liczba | Suma | Średnia")
        
        for s, liczba in store_counts.items():
            suma = total_amount[s]
            srednia = suma / liczba
            print(f"{s} | {liczba} | {suma:.2f} | {srednia:.2f}")

Writing consumer_count.py


In [5]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {'liczba': 0, 'suma': 0.0, 'min': 999999.0, 'max': 0.0})
msg_count = 0


for message in consumer:
    dane = message.value
    kategoria = dane['category']
    kwota = dane['amount']
    
    stats[kategoria]['liczba'] += 1
    stats[kategoria]['suma'] += kwota
    
    if kwota < stats[kategoria]['min']:
        stats[kategoria]['min'] = kwota
        
    if kwota > stats[kategoria]['max']:
        stats[kategoria]['max'] = kwota
        
    msg_count += 1
    
    if msg_count % 10 == 0:
        print(f"\n--- STATYSTYKI KATEGORII (po {msg_count} transakcjach) ---")
        print("Kategoria | Liczba | Suma | Min | Max")
        
        for kat, s in stats.items():
            print(f"{kat} | {s['liczba']} | {s['suma']:.2f} | {s['min']:.2f} | {s['max']:.2f}")

Writing consumer_stats.py


In [6]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
import json, time
from collections import defaultdict

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

historia_czasow = defaultdict(list)

print("Czekam na oszustów (więcej niż 3 transakcje w 60 sekund)...")

for message in consumer:
    dane = message.value
    user = dane['user_id']

    teraz = time.time() 

    historia_czasow[user].append(teraz)

    historia_czasow[user] = [czas for czas in historia_czasow[user] if teraz - czas <= 60]
    
    if len(historia_czasow[user]) > 3:
        print(f"ALARM! {user} wariuje! Zrobił {len(historia_czasow[user])} transakcje w minutę!")

Writing consumer_anomaly.py
